# PARC2026 — 73 M3 A100 training smoke

Notebook 72d PASS後の **training smoke-only** 工程です。72dの実micro-batch/gradient-accumulationを使い、同一training schedule seedでforward/reverse × equal-data/equal-wall × 3モデルを短時間だけ実行します。OpenVLAの7B mergeはここでは行わず、評価時にのみ一時materializeします。1800秒benchmarkは開始しません。


In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is required; add it to Colab Secrets as HF_TOKEN')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_training_smoke'
PIN = '6e907ef5663f37b85010d0d1620ebc45d539105e'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('73 smoke code:', got, flush=True)
env = os.environ.copy()
env['PY_AI_REPO'] = str(REPO)
env['PARC_ROOT'] = str(ROOT)
env['PARC_DRIVE_ROOT'] = '/content/drive/MyDrive/parc2026-cache'
env['PARC_M3_EXECUTE'] = '1'  # explicit smoke opt-in; runner cannot launch benchmark mode
subprocess.run(['python', '-u', str(REPO / 'tools/colab/run_m3_training_smoke.py')], cwd=str(REPO), env=env, check=True)
print('=== 73 COMPLETE ===', flush=True)
print('Training smoke only. Full 1800-second M3 benchmark has NOT started.', flush=True)
